# Viga interactiva: de la librería a las diapositivas

Hoy el mismo cálculo de la viga viaja por tres archivos de **texto**:

1. **`vigas.py`**, la librería: las fórmulas, escritas una sola vez.
2. **Este notebook**: prueba la librería, la convierte en una mini interfaz y exporta los resultados.
3. **`../slides/data/viga_datos.js`**: los datos que lee la presentación para dibujar la viga en directo.

Para verlo todo junto, abre en VS Code la carpeta **`ProfesIA` entera** (no solo `ejemplos`).

In [1]:
# Si editas vigas.py (tú o la IA), el notebook recoge los cambios sin reiniciar
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
from vigas import (
    APOYOS,
    PERFILES,
    calcula_viga,
    exporta_para_diapositivas,
    tension_maxima,
)

## 1. Datos (unidades del SI)

In [2]:
L = 3.0  # luz [m]
E = 210e9  # acero [Pa]
FY = 275e6  # límite elástico del acero S275 [Pa]
PERFIL = "IPE 200"  # prueba también "IPE 160", "IPE 240" o "IPE 300"

PERFILES[PERFIL]

{'I': 1.943e-05, 'W': 0.0001943, 'h': 200}

## 2. Un caso: la viga del primer día

Voladizo de 3 m con 10 kN en el extremo. Tiene que salir lo mismo que `viga.py`: **22.1 mm**.

In [3]:
x = np.linspace(0, L, 201)
seccion = PERFILES[PERFIL]

flecha, momento = calcula_viga(x, L, a=L, P=10e3, E=E, I=seccion["I"], apoyo="voladizo")
sigma = tension_maxima(momento, seccion["W"])

print(f"Flecha máxima:  {flecha.max() * 1000:.1f} mm")
print(f"Momento máximo: {np.abs(momento).max() / 1e3:.1f} kN·m")
print(f"Tensión máxima: {sigma / 1e6:.0f} MPa  ({sigma / FY:.0%} del límite elástico)")

Flecha máxima:  22.1 mm
Momento máximo: 30.0 kN·m
Tensión máxima: 154 MPa  (56% del límite elástico)


## 3. Una mini interfaz con `ipywidgets`

Mueve los deslizadores: dónde está la carga, cuánto vale, cómo se apoya la viga y qué perfil usamos.
Arriba, la deformada (exagerada); abajo, el momento flector: **la viga se dobla más donde el momento es mayor**.

In [ ]:
import ipywidgets as widgets


def dibuja_viga(a=3.0, P=10.0, apoyo="voladizo", perfil=PERFIL):
    s = PERFILES[perfil]
    w, M = calcula_viga(x, L, a, P * 1e3, E, s["I"], apoyo)
    sigma = tension_maxima(M, s["W"])
    i = np.abs(M).argmax()
    color = "tab:red" if sigma > FY else "tab:green"

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
    wmax = max(w.max() * 1000, 0.1)  # mm
    ax1.plot(x, -w * 1000, color=color, lw=3)
    ax1.axhline(0, color="0.7", ls="--")
    ax1.annotate(
        f"{P:.0f} kN",
        xy=(a, 0),
        xytext=(a, 0.6 * wmax),
        ha="center",
        arrowprops=dict(arrowstyle="->", lw=2),
    )
    ax1.set_ylim(-1.2 * wmax, 0.9 * wmax)
    ax1.set_ylabel("flecha [mm]")
    ax1.set_title(
        f"{apoyo}, {perfil}:  flecha máx. {w.max() * 1000:.1f} mm  ·  "
        f"σ máx. {sigma / 1e6:.0f} MPa ({sigma / FY:.0%} de fy)",
        color=color,
    )

    ax2.fill_between(x, M / 1e3, color=color, alpha=0.3)
    ax2.plot(x, M / 1e3, color=color)
    ax2.plot(x[i], M[i] / 1e3, "o", color="k")
    ax2.annotate(
        "aquí se dobla más",
        (x[i], M[i] / 1e3),
        xytext=(8, 8),
        textcoords="offset points",
    )
    ax2.set_xlabel("x [m]")
    ax2.set_ylabel("momento [kN·m]")
    for ax in (ax1, ax2):
        ax.grid(True, alpha=0.3)
    plt.show()


widgets.interact(
    dibuja_viga,
    a=widgets.FloatSlider(value=L, min=0, max=L, step=0.1, description="a [m]"),
    P=widgets.FloatSlider(value=10, min=0, max=40, step=1, description="P [kN]"),
    apoyo=widgets.ToggleButtons(options=APOYOS),
    perfil=widgets.Dropdown(options=list(PERFILES), value=PERFIL),
);

interactive(children=(FloatSlider(value=3.0, description='a [m]', max=3.0), FloatSlider(value=10.0, descriptio…

## 4. Exportar para las diapositivas

Esta celda escribe `slides/data/viga_datos.js`: un archivo de texto con las curvas de la viga
para el `PERFIL` elegido arriba. La diapositiva «La viga, en directo» lo lee y dibuja la viga
mientras arrastras la carga con el ratón.

In [5]:
ruta = exporta_para_diapositivas(
    "../slides/data/viga_datos.js", L=L, E=E, perfil=PERFIL, fy=FY
)
print(f"Escrito {ruta}  ({ruta.stat().st_size / 1024:.0f} kB)")
print(ruta.read_text()[:300], "…")

Escrito ../slides/data/viga_datos.js  (57 kB)
// Archivo generado por ejemplos/viga_interactiva.ipynb con vigas.py.
// No lo edites a mano: cambia el notebook y vuelve a ejecutarlo.
window.VIGA_DATOS = {"generado": "18/11/2026 10:05", "fuente": "ejemplos/viga_interactiva.ipynb", "L": 3.0, "E": 210000000000.0, "fy": 275000000.0, "perfil": {"nomb …


## 5. Reto

1. Cambia `PERFIL` a `"IPE 300"` en la celda de datos y ejecuta **todo** el notebook (*Run All*).
2. Vuelve a la presentación y recarga la página: la viga de la diapositiva ya es un IPE 300.
3. Pídele a la IA: *«Añade a `vigas.py` el caso de una viga biempotrada y que salga también en las diapositivas»*.
   Mira qué archivos toca: la librería, el notebook… y la presentación.